In [0]:
!pip install transformers=4.57.3
!pip install torchvision
!pip install torch
!pip install accelerate
!pip install lm_eval
!pip install datasets
!pip install tqdm


In [0]:
import transformers
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, AutoModelForCausalLM
import tqdm
import logging
from tqdm import tqdm
import datasets
from lm_eval import evaluator
from lm_eval.models.huggingface import HFLM
from lm_eval.utils import make_table
from transformers import PreTrainedTokenizerBase
from torch.utils.data import DataLoader, Dataset, SubsetRandomSampler
print(transformers.__version__)

import warnings
warnings.filterwarnings("ignore", module="datasets")

In [0]:
key =""

In [0]:
model_name = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=False)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    token = key
    # device_map="auto"
)

In [0]:
class InterruptExecution(Exception):
    pass

class algo1_functor_mod():
    def __init__(self, model , interim):
        self.co_var = torch.zeros(interim, interim).to(model.device)
        # self.scores = torch.zeros(interim).to(model.device)
    
    def __call__(self,module,input):
        # scores = torch.square(input[0]).sum(dim=(0,1))/input[0].shape[0]
        # self.scores.add_(scores)
        torch.addbmm(self.co_var,input[0].transpose(1,2),input[0]/input[0].shape[0],out = self.co_var)
        # torch.addmm(self.co_var,input[0].transpose(0,1),input[0]/input[0].shape[0],out = self.co_var)
        raise InterruptExecution()

In [0]:
class Prune():
    def __init__(self):
        pass

    def prune(self, model, train_loader, sparsity_layer):
        
        list_layers = list(model.model.layers.named_children())

        storage_pos = {}

        def prehook_pos_emb(module,args,kwargs):
            storage_pos["positional_embeddings"] = kwargs.get("position_embeddings")
            
        pre_hook = model.model.layers[0].register_forward_pre_hook(prehook_pos_emb,with_kwargs = True)
        for layer in tqdm(range(model.config.num_hidden_layers)):
            if layer>=10 and layer<=20:
                module = list_layers[layer][1].mlp
                interim = module.gate_proj.weight.shape[0]
                name = list_layers[layer][0]
                module._name = name
                functor = algo1_functor_mod(model, interim)
                hook = module.down_proj.register_forward_pre_hook(functor)
                # hook = module.register_forward_hook(functor)
                
                for batch in train_loader:
                    batch.pop("labels")
                    batch = {k: v.to(model.device) for k, v in batch.items()}
                    with torch.no_grad():
                        try:
                            _ = model(**batch)
                        except InterruptExecution:
                            pass
                
                hook.remove()
                ## pruning
                torch.cuda.empty_cache()
                co_var = functor.co_var
                # co_var = co_var.to("cuda:0")
                
                
                scores_vector = torch.diagonal(co_var)

                ## WANDA
                norm = torch.norm(module.down_proj.weight,dim=0)
                scores_vector = scores_vector*norm
                
                k = int((1-sparsity_layer[int(module._name)])*co_var.size(0))
                topk_scores, topk_indices = torch.topk(scores_vector, k, largest=True)

                ##random pruning
                # topk_indices = torch.randint(0, len(scores_vector), size=(k,))

                topk_indices = topk_indices.sort().values
                d_int = scores_vector.shape[0]

                S_k = torch.zeros((d_int, k), dtype=torch.float32)
                S_k[topk_indices, torch.arange(k)] = 1.0

                W_U_k = torch.concat([module.up_proj.weight,module.gate_proj.weight],dim=1).T.to(torch.float32) @ S_k.to(module.gate_proj.weight.device) 
                co_var = co_var.to("cpu")

                W_D_k = module.down_proj.weight@S_k.to(module.down_proj.weight.device)
                
                
                new_up_proj = W_U_k.T[:,:model.config.hidden_size]
                new_gate_proj = W_U_k.T[:,model.config.hidden_size:]
                module.up_proj.weight.data = new_up_proj.to(module.up_proj.weight.device)
                module.gate_proj.weight.data = new_gate_proj.to(module.gate_proj.weight.device)

                module.down_proj.weight.data = W_D_k.to(module.down_proj.weight.device)
        pre_hook.remove()


In [0]:
def format_prompt_sciq(example):
    text = f"{example['support'].lstrip()}\nQuestion: {example['question']}\nAnswer:"
    example["text"] = text
    return example
ds = datasets.load_dataset("allenai/sciq")
ds = ds.map(format_prompt_sciq,load_from_cache_file=False,   keep_in_memory=True )
ds = ds.remove_columns(['question','distractor3','distractor2','distractor1','correct_answer','support'])

train_dataset_sci, test_dataset_sci = ds["train"].shuffle(seed=42).select(range(5000)), ds["test"].shuffle(seed=42).select(range(300))

In [0]:
def prepare_dataloader(
    dataset: datasets.Dataset,
    tokenizer: PreTrainedTokenizerBase,
    max_seqlen: int = 2048,
    batch_size: int = 1,
    nsamples: int = 128,
    varied_seqlen: bool = False,
    seed=42,
) -> DataLoader[dict[str, torch.Tensor]]:
    """
    Get a DataLoader from a dataset.

    Args:
        dataset: The dataset to create a dataloader from.
        tokenizer: The tokenizer to use.
        max_seqlen: The maximum sequence length, used for truncation of sequences in the dataset.
        batch_size: The batch size.
        nsamples: The number of samples to produce.
        varied_seqlen: If False, concatenate multiple examples from the dataset into one example until max_seqlen is reached.
        seed: The seed for sampling the dataset.

    Returns:
        A DataLoader.
    """
    logging.info(f"Preparing dataloader")

    if not varied_seqlen and not nsamples:
        logging.warning(
            "varied_seqlen=False, but nsamples is not specified. This will lead to tokenization of the entire dataset, which will be slow."
        )
    data_name = dataset.column_names[0]
    ds = dataset.filter(lambda x: len(x[data_name]) > 0)

    if not varied_seqlen:
        # create a new dataset where each example is a concatenation of multiple examples of total length = max_seqlen.
        data_list = ds[data_name]
        new_data_list = []

        torch.manual_seed(seed)
        indices = list(range(len(data_list)))

        while len(new_data_list) < nsamples and len(indices) > 0:
            start_idx = torch.randint(0, len(indices), (1,)).item()
            idx = start_idx
            tokens = []
            while len(tokens) < max_seqlen and idx < len(indices):
                item = data_list[indices[idx]]
                sep = "" if not tokens else "\n\n"
                tokens += tokenizer.tokenize(sep + item)
                idx += 1

            indices = indices[:start_idx] + indices[idx:]  # remove the used indices

            if len(tokens) >= max_seqlen:
                tokens = tokens[:max_seqlen]  # truncate to max_seqlen
                new_data_list.append(tokenizer.convert_tokens_to_string(tokens))

        ds = datasets.Dataset.from_dict({data_name: new_data_list})
        def tokenize(data_batch):
        # tokenize then pad each batch according to the longest sequence in the batch
            batch = tokenizer(
                data_batch[data_name],
                padding="longest",
                max_length=max_seqlen,
                truncation=True,
                return_tensors="pt",
            )
            batch["labels"] = batch["input_ids"].clone()
            return batch

    # tokenize lazily
    ds.set_transform(tokenize)

    torch.manual_seed(seed)
    sampler = SubsetRandomSampler(torch.randperm(len(ds))[:nsamples])

    loader = DataLoader(ds, batch_size=batch_size, sampler=sampler)
    logging.info(f"Preparing dataloader done")
    return loader


train_loader = prepare_dataloader(
dataset=train_dataset_sci,
tokenizer=tokenizer,
max_seqlen=512,
batch_size=32,
nsamples=128,
varied_seqlen=False,
seed=42)

In [0]:
sparsity_layer = []
for i in range(model.config.num_hidden_layers):
    sparsity_layer.append(0.5)

In [0]:
datasets.utils.logging.disable_progress_bar()
print(sum(int(p.nelement()) for p in model.parameters()))

# lm = HFLM(pretrained=model, tokenizer=tokenizer)
# results = evaluator.simple_evaluate(model=lm,tasks=["sciq"],batch_size=1,limit =100)
# print(make_table(results))


task_masker = Prune()
task_masker.prune(model, train_loader, sparsity_layer)

print(sum(int(p.nelement()) for p in model.parameters()))

lm = HFLM(pretrained=model, tokenizer=tokenizer)
results = evaluator.simple_evaluate(model=lm,tasks=["sciq"],batch_size=1,limit =100)
print(make_table(results))

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    token = key
    # device_map="auto"
)

In [0]:
import mlflow

# Log your pruned Qwen model
with mlflow.start_run():
    mlflow.transformers.log_model(
        transformers_model={"model": model, "tokenizer": tokenizer},
        artifact_path="qwen_pruned_real",
        registered_model_name="qwen_pruned_model"
    )

In [0]:
client = mlflow.MlflowClient()
versions = client.get_latest_versions("qwen_dense_model")
for v in versions:
    print(f"Version: {v.version}")
    model_info = mlflow.models.get_model_info(f"models:/qwen_dense_model/{v.version}")
    print(model_info.signature)

In [0]:
import mlflow
from mlflow.tracking import MlflowClient

client = MlflowClient()

# List versions in UC
versions = client.search_model_versions("name='qwen_dense_model'")
for v in versions:
    print(f"Version: {v.version}")
    model_info = mlflow.models.get_model_info(f"models:/qwen_dense_model/{v.version}")
    print(model_info.signature)